# Using Your AIS Data

Not every project starts from a database that someone else has already built. Often you have a raw AIS file, either your own receiver logs or an open-source archive, and you need to turn it into queryable tracks. This notebook walks that path end to end using a real open dataset: it downloads a full day of United States AIS from NOAA, decodes it straight into a SQLite database with AISdb, queries a Gulf of Mexico window, reshapes the rows into per-vessel tracks, and prints a short summary.

## What you will learn

- How to fetch an open NOAA daily AIS archive without leaving Python
- How `aisdb.decode_msgs` reads the raw NOAA CSV directly when you pass `source='NOAA'`, so no manual column remapping is needed
- How to query a decoded database by time and bounding box with `DBQuery`
- How `TrackGen` collapses raw rows into one dictionary per vessel
- How to summarize the result with a vessel count and a position count

This notebook is the runnable companion to the [Using Your AIS Data](https://aisviz.gitbook.io/documentation/tutorials/using-your-ais-data) page in the AISViz documentation. The same Gulf of Mexico window and NOAA day file power the Machine Learning section, so the database you build here is a valid input for those tutorials too.

In [ ]:
# If AISdb is not already installed, uncomment the line below.
# On Google Colab, run it once and then restart the runtime.
# %pip install aisdb

# This notebook uses only AISdb plus the Python standard library.

## Download the NOAA day file

NOAA publishes daily archives of United States AIS at [coast.noaa.gov](https://coast.noaa.gov/htdata/CMSP/AISDataHandler/). We use the archive for 2020-01-01, the same day the Machine Learning tutorials build on, so the database you create here is reproducible and reusable.

The download is a single zip of about 260 MB (264 MB on the wire, roughly 1.9 GB once decoded into the database). That is large enough that you do not want to fetch it twice, so the cell below checks whether the file already exists and only downloads when it is missing. Re-running the notebook after the first successful download skips the transfer entirely.

We use `urllib.request.urlretrieve` from the standard library rather than a shell command so the notebook runs the same way on every platform, including Colab and Windows.

In [ ]:
import urllib.request
from pathlib import Path

url = 'https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2020/AIS_2020_01_01.zip'
zip_path = Path('./data/AIS_2020_01_01.zip')
zip_path.parent.mkdir(parents=True, exist_ok=True)

# The archive is ~260 MB. Skip the download if it is already on disk.
if zip_path.exists():
    print(f'Already downloaded: {zip_path} ({zip_path.stat().st_size / 1e6:.0f} MB)')
else:
    print('Downloading ~260 MB from NOAA, this can take a few minutes ...')
    urllib.request.urlretrieve(url, zip_path)
    print(f'Saved: {zip_path} ({zip_path.stat().st_size / 1e6:.0f} MB)')

## Decode straight into SQLite

AISdb knows the NOAA CSV layout, so you do not have to rename columns or reshape the file by hand. Passing `source='NOAA'` tells `decode_msgs` to read the raw MarineCadastre columns (`BaseDateTime`, `LAT`, `LON`, `VesselType`, and the rest) and map them internally. It also unpacks the zip for you, so a single call covers the whole download-to-database step.

The decoded messages land in month-stamped tables (`ais_202001_dynamic`, `ais_202001_static`, and a `static_202001_aggregate` view) inside the SQLite file. Decoding a full national day takes a couple of minutes and writes roughly 1.9 GB, so, like the download, this step is worth doing once and then reusing the database.

## Work on a sample first

One day of NOAA data holds around seven million messages, and decoding all of them takes several minutes. For a first pass we extract the CSV from the archive and keep the first 300,000 rows, which decodes in seconds and behaves identically. Flip `FULL_DAY` to `True` when you want the complete day.

In [ ]:
import zipfile

FULL_DAY = False  # set True to decode the full ~7M message day

if FULL_DAY:
    decode_path = zip_path
else:
    sample_path = zip_path.parent / 'AIS_2020_01_01_sample.csv'
    if not sample_path.exists():
        with zipfile.ZipFile(zip_path) as archive:
            csv_name = next(n for n in archive.namelist() if n.endswith('.csv'))
            with archive.open(csv_name) as full, open(sample_path, 'wb') as out:
                for i, line in enumerate(full):
                    if i > 300_000:
                        break
                    out.write(line)
    print(f'sample ready: {sample_path} ({sample_path.stat().st_size / 1e6:.0f} MB)')
    decode_path = sample_path

In [ ]:
import aisdb
from aisdb import SQLiteDBConn

dbpath = './data/noaa_2020_01_01.db'

# Decode the raw NOAA zip directly into a SQLite database.
# AISdb skips files whose checksum it has already ingested, so re-running is safe.
with SQLiteDBConn(dbpath=dbpath) as dbconn:
    aisdb.decode_msgs(
        filepaths=[str(decode_path)],
        dbconn=dbconn,
        source='NOAA',
        verbose=True,
    )

## Query a Gulf of Mexico window

A national day file holds millions of messages, far more than any single analysis needs. `DBQuery` lets you pull back just a time range and a geographic box. Here we take the whole of 2020-01-01 and the Gulf of Mexico bounding box that spans the Texas and Florida coasts, matching the window the Machine Learning section uses so the tracks line up across tutorials.

The `xmin`, `xmax`, `ymin`, and `ymax` arguments define the box in degrees, and `in_bbox_time_validmmsi` is the callback that filters rows to that box and time range while dropping malformed MMSIs. `gen_qry()` yields rows sorted by MMSI and time, and `TrackGen` collapses those rows into one dictionary per vessel. We pass `decimate=False` to keep every position report, since the summary below counts raw positions.

In [ ]:
from datetime import datetime
from aisdb import DBQuery
from aisdb.database import sqlfcn_callbacks

with SQLiteDBConn(dbpath=dbpath) as dbconn:
    qry = DBQuery(
        dbconn=dbconn,
        start=datetime(2020, 1, 1),
        end=datetime(2020, 1, 2),
        xmin=-98, xmax=-80, ymin=24, ymax=31,
        callback=sqlfcn_callbacks.in_bbox_time_validmmsi,
    )
    rowgen = qry.gen_qry()
    tracks = list(aisdb.track_gen.TrackGen(rowgen, decimate=False))

# Summarize: how many distinct vessels, and how many position reports in total.
vessel_count = len(tracks)
position_count = sum(track['time'].size for track in tracks)

print(f'Vessels:   {vessel_count:,}')
print(f'Positions: {position_count:,}')

Each entry in `tracks` is a plain dictionary. `track['mmsi']` identifies the vessel, `track['time']` holds epoch seconds, and `track['lon']` and `track['lat']` hold degrees, the same arrays every other AISdb tutorial works with. From here the tracks are ready for cleaning, interpolation, export, or a map, exactly as the querying and cleaning tutorials describe.

If you are working against a PostgreSQL or TimescaleDB backend instead of SQLite, swap `SQLiteDBConn` for `PostgresDBConn(hostaddr=..., port=..., user=..., dbname=..., password=...)`. Both connection classes implement the same `DBConn` interface, so the query and track-generation code above stays the same.

## Next steps

- Clean the tracks you just built with the denoising helpers in the [Data Cleaning](https://aisviz.gitbook.io/documentation/tutorials/data-cleaning) tutorial.
- Interpolate and export them following [Track Interpolation](https://aisviz.gitbook.io/documentation/tutorials/track-interpolation).
- Read the full narrative, including the PostgreSQL and bulk-download pipelines, on the [Using Your AIS Data](https://aisviz.gitbook.io/documentation/tutorials/using-your-ais-data) documentation page.